# Preprocessing & Binary Anxiety Classification with STAI Questionnaire Labels

This notebook preprocesses WESAD signals and extracts 60-second windows every 12 seconds. For each participant, it reverse-scores the WESAD shortened STAI items (items 1, 4, and 6) and averages the resulting valid responses by condition and labels a non-Base condition Anxiety (`1`) when its mean is at least 0.2 points above that participant’s Base mean; otherwise it labels No Anxiety (`0`). This is an operational, baseline-relative self-report label: it does not assume that TSST must increase every participant’s STAI score. Whether TSST reliably increases STAI is a separate empirical hypothesis to evaluate from the participant-level scores. Only windows fully contained in one timed, labeled condition are retained. Existing `S*_stai.pkl` files must be regenerated after changing the labeling logic or threshold.

**ACC sub-window variant:** each retained 60-second main window also computes ACC-specific features over 5-second sub-windows shifted by 0.25 seconds. The resulting ACC statistics are aggregated using the mean and standard deviation across sub-windows, while all non-ACC signals keep the original 60-second processing window.


In [1]:
import gc
import pickle
import sys
import time
import warnings
from pathlib import Path
from unittest.mock import MagicMock

import matplotlib
matplotlib.use('Agg')
import numpy as np
import pandas as pd
import heartpy as hp
import neurokit2 as nk

warnings.simplefilter('ignore')

# Prevent biosppy's optional interactive plotting modules from requiring tkinter.
sys.modules['biosppy.inter_plotting'] = MagicMock()
sys.modules['biosppy.inter_plotting.ecg'] = MagicMock()
sys.modules['biosppy.inter_plotting.acc'] = MagicMock()
import biosppy

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'WESAD').exists())
FB_CODE_DIR = REPO_ROOT / '_forked' / 'fb_code'
if str(FB_CODE_DIR) not in sys.path:
    sys.path.insert(0, str(FB_CODE_DIR))

from utils import (
    butter_lowpass_filter,
    filterSignalFIR as filter_signal_fir,
    get_net_accel,
    get_window_stats,
    get_peak_freq,
    get_absolute_integral,
    get_slope,
    get_dynamic_range,
)
from respiration import get_resp_features

# User-configurable pipeline settings.
SUBJECTS = [2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 13, 14, 15, 16, 17]
WINDOW_LENGTH_SECONDS = 60
WINDOW_SHIFT_SECONDS = 12  # shorter shift for denser coverage of brief conditions
ACC_WINDOW_LENGTH_SECONDS = 5
ACC_WINDOW_SHIFT_SECONDS = 0.25
STAI_INCREASE_THRESHOLD = 0.2  # anxiety if condition mean - personal Base mean >= this
SKIP_EXTRACTED_SUBJECTS = False  # existing files contain old protocol-derived labels
OVERWRITE_EXISTING_OUTPUTS = True  # set False to refuse replacing existing files

FS = {'ACC': 32, 'BVP': 64, 'EDA': 4, 'TEMP': 4, 'label': 700,
      'Resp': 700, 'ECG': 700, 'chest': 700}
WESAD_DIR = REPO_ROOT / 'WESAD'
OUTPUT_DIR = REPO_ROOT / 'Data' / 'Preprocessed n STAI-labeled'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = REPO_ROOT / 'Models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

if WINDOW_LENGTH_SECONDS <= 0 or WINDOW_SHIFT_SECONDS <= 0:
    raise ValueError('WINDOW_LENGTH_SECONDS and WINDOW_SHIFT_SECONDS must be positive.')
if ACC_WINDOW_LENGTH_SECONDS <= 0 or ACC_WINDOW_SHIFT_SECONDS <= 0:
    raise ValueError('ACC_WINDOW_LENGTH_SECONDS and ACC_WINDOW_SHIFT_SECONDS must be positive.')
if ACC_WINDOW_LENGTH_SECONDS > WINDOW_LENGTH_SECONDS:
    raise ValueError('ACC_WINDOW_LENGTH_SECONDS cannot exceed WINDOW_LENGTH_SECONDS.')
print('Setup completed:', OUTPUT_DIR)
print(f'Window length={WINDOW_LENGTH_SECONDS}s, shift={WINDOW_SHIFT_SECONDS}s')
print(f'ACC sub-window length={ACC_WINDOW_LENGTH_SECONDS}s, shift={ACC_WINDOW_SHIFT_SECONDS}s')
print(f'STAI increase threshold={STAI_INCREASE_THRESHOLD}')

Setup completed: e:\Rizqy\Kuliah\Skripsi\repo-skripsi\Data\Preprocessed n STAI-labeled
Window length=60s, shift=12s
ACC sub-window length=5s, shift=0.25s
STAI increase threshold=0.2


## 1. Questionnaire & STAI Parser

In [2]:
def parse_clock_mmss(value):
    """Convert questionnaire MM.SS notation (e.g. 7.08 -> 428 seconds)."""
    text = str(value).strip()
    if not text:
        return np.nan
    if '.' in text:
        minute_text, second_text = text.split('.', 1)
        minutes = int(minute_text)
        seconds = int(second_text)
    else:
        minutes, seconds = int(text), 0
    if not 0 <= seconds < 60:
        raise ValueError(f'Invalid MM.SS time {value!r}: seconds must be 00-59')
    return minutes * 60 + seconds


def _parse_numeric_fields(parts):
    """Preserve CSV field alignment, converting empty fields to NaN."""
    values = []
    for part in parts:
        text = part.strip()
        if not text:
            values.append(np.nan)
        else:
            try:
                values.append(float(text))
            except ValueError:
                values.append(np.nan)
    return values


# WESAD uses a shortened six-item state-anxiety questionnaire. The positively
# worded items are reverse-scored before averaging: 1 <-> 4 and 2 <-> 3.
STAI_ITEM_NAMES = (
    'feel_at_ease', 'feel_nervous', 'jittery', 'relaxed', 'worried', 'pleasant'
)
STAI_REVERSE_ITEM_INDICES = {0, 3, 5}


def score_stai_responses(raw_responses):
    """Return raw and correctly reverse-scored six-item STAI responses."""
    raw = [float(value) if pd.notna(value) else np.nan for value in raw_responses]
    if len(raw) > len(STAI_ITEM_NAMES):
        raise ValueError(f'Expected at most {len(STAI_ITEM_NAMES)} shortened STAI items, got {len(raw)}')
    scored = [
        (5.0 - value) if index in STAI_REVERSE_ITEM_INDICES and pd.notna(value) else value
        for index, value in enumerate(raw)
    ]
    return raw, scored


def parse_quest_file(subject_id):
    quest_path = WESAD_DIR / f'S{subject_id}' / f'S{subject_id}_quest.csv'
    if not quest_path.exists():
        raise FileNotFoundError(f'Questionnaire file not found: {quest_path}')

    order = start_raw = end_raw = None
    stai_rows = []
    with open(quest_path, 'r', encoding='latin1') as f:
        for line in f:
            parts = [part.strip() for part in line.rstrip('\r\n').split(';')]
            tag = parts[0].strip() if parts else ''
            if tag.startswith('# ORDER'):
                order = [value.strip() for value in parts[1:]]
            elif tag.startswith('# START'):
                start_raw = parts[1:]
            elif tag.startswith('# END'):
                end_raw = parts[1:]
            elif tag.startswith('# STAI'):
                stai_rows.append(_parse_numeric_fields(parts[1:]))

    if order is None or start_raw is None or end_raw is None:
        raise ValueError(f'{quest_path.name}: missing ORDER, START, or END row')
    if len(start_raw) < len(order) or len(end_raw) < len(order):
        raise ValueError(f'{quest_path.name}: timing rows do not cover all conditions')
    if not stai_rows:
        raise ValueError(f'{quest_path.name}: no STAI response rows found')

    conditions = []
    for index, condition in enumerate(order):
        if not condition:
            continue
        start_text, end_text = start_raw[index].strip(), end_raw[index].strip()
        start_seconds = parse_clock_mmss(start_text) if start_text else np.nan
        end_seconds = parse_clock_mmss(end_text) if end_text else np.nan
        responses = [row[index] if index < len(row) else np.nan for row in stai_rows]
        raw_responses, scored_responses = score_stai_responses(responses)
        valid_raw = [float(value) for value in raw_responses if pd.notna(value)]
        valid_scored = [float(value) for value in scored_responses if pd.notna(value)]
        if any(value < 1 or value > 4 for value in valid_raw):
            raise ValueError(f'{quest_path.name}: STAI response outside 1-4 for condition {condition!r}')
        conditions.append({
            'condition': condition,
            'start_seconds': start_seconds,
            'end_seconds': end_seconds,
            'stai_responses_raw': valid_raw,
            'stai_responses_scored': valid_scored,
            'stai_item_count': len(valid_scored),
            'stai_mean_raw': float(np.mean(valid_raw)) if valid_raw else np.nan,
            'stai_mean': float(np.mean(valid_scored)) if valid_scored else np.nan,
        })

    if not any(item['condition'].strip().lower() == 'base' for item in conditions):
        raise ValueError(f'{quest_path.name}: no Base condition found')
    return conditions


# Inspect parsed condition data for every configured participant.
all_participant_conditions = []
for subject_id in SUBJECTS:
    participant_df = pd.DataFrame(parse_quest_file(subject_id))
    participant_df.insert(0, 'subject_id', subject_id)
    all_participant_conditions.append(participant_df)

all_conditions_df = pd.concat(all_participant_conditions, ignore_index=True)
display(all_conditions_df.sort_values(['subject_id', 'condition']).reset_index(drop=True))


,subject_id,condition,start_seconds,end_seconds,stai_responses_raw,stai_responses_scored,stai_item_count,stai_mean_raw,stai_mean
0,2,Base,428,1592,"[3.0, 1.0, 4.0, 3.0, 4.0]","[2.0, 1.0, 4.0, 2.0, 4.0]",5,3.0,2.6
1,2,Fun,4885,5267,"[4.0, 1.0, 4.0, 3.0, 4.0]","[1.0, 1.0, 4.0, 2.0, 4.0]",5,3.2,2.4
2,2,Medi 1,4219,4621,"[1.0, 2.0, 1.0, 1.0, 1.0]","[4.0, 2.0, 1.0, 4.0, 1.0]",5,1.2,2.4
3,2,Medi 2,5618,6015,"[2.0, 3.0, 2.0, 1.0, 1.0]","[3.0, 3.0, 2.0, 4.0, 1.0]",5,1.8,2.6
4,2,TSST,2395,3003,"[2.0, 3.0, 1.0, 1.0, 1.0]","[3.0, 3.0, 1.0, 4.0, 1.0]",5,1.6,2.4
...,...,...,...,...,...,...,...,...,...
114,17,Medi 2,5349,5763,"[1.0, 1.0, 1.0, 4.0, 1.0]","[4.0, 1.0, 1.0, 1.0, 1.0]",5,1.6,1.6
115,17,TSST,3602,4345,"[2.0, 4.0, 3.0, 1.0, 2.0]","[3.0, 4.0, 3.0, 4.0, 2.0]",5,2.4,3.2
116,17,bRead,1417,1495,"[2.0, 4.0, 2.0, 1.0, 1.0]","[3.0, 4.0, 2.0, 4.0, 1.0]",5,2.0,2.8
117,17,fRead,2282,2359,[],[],0,NaN,NaN


## 2. Signal Preprocessing & Binary STAI Labeling Pipeline

In [3]:
def load_raw_subject(subject_id):
    path = WESAD_DIR / f'S{subject_id}' / f'S{subject_id}.pkl'
    with open(path, 'rb') as file:
        return pickle.load(file, encoding='latin1')


def build_condition_labels(subject_id):
    conditions = parse_quest_file(subject_id)
    base = next(item for item in conditions if item['condition'].strip().lower() == 'base')
    if pd.isna(base['stai_mean']):
        raise ValueError(f'S{subject_id}: Base STAI mean cannot be calculated')
    base_mean = base['stai_mean']

    labeled = []
    for item in conditions:
        name = item['condition']
        if name.strip().lower() == 'base':
            continue
        if pd.isna(item['stai_mean']) or pd.isna(item['start_seconds']) or pd.isna(item['end_seconds']):
            continue
        if item['end_seconds'] <= item['start_seconds']:
            raise ValueError(f'S{subject_id}: invalid interval for {name}: {item}')
        item = dict(item)
        item['baseline_stai_mean'] = base_mean
        item['binary_anxiety'] = int(item['stai_mean'] - base_mean >= STAI_INCREASE_THRESHOLD)
        labeled.append(item)

    if not labeled:
        raise ValueError(f'S{subject_id}: no non-Base conditions have both valid STAI scores and timing')
    return labeled


def compute_features_stai(subject_id):
    """Filter and align raw physiological channels to the WESAD 700-Hz timeline."""
    raw_data = load_raw_subject(subject_id)
    wrist = raw_data['signal']['wrist']
    chest = raw_data['signal']['chest']
    labels = np.asarray(raw_data['label']).ravel()

    eda_df = pd.DataFrame(wrist['EDA'], columns=['EDA'])
    bvp_df = pd.DataFrame(wrist['BVP'], columns=['BVP'])
    acc_df = pd.DataFrame(wrist['ACC'], columns=['ACC_x', 'ACC_y', 'ACC_z'])
    temp_df = pd.DataFrame(wrist['TEMP'], columns=['TEMP'])
    acc_c_df = pd.DataFrame(chest['ACC'], columns=['ACC_x_C', 'ACC_y_C', 'ACC_z_C'])
    ecg_df = pd.DataFrame(chest['ECG'], columns=['ECG'])
    eda_c_df = pd.DataFrame(chest['EDA'], columns=['EDA_C'])
    emg_c_df = pd.DataFrame(chest['EMG'], columns=['EMG_C'])
    resp_c_df = pd.DataFrame(chest['Resp'], columns=['Resp_C'])
    temp_c_df = pd.DataFrame(chest['Temp'], columns=['Temp_C'])

    eda_df['EDA'] = butter_lowpass_filter(eda_df['EDA'], 1.0, FS['EDA'], 6)
    eda_c_df['EDA_C'] = butter_lowpass_filter(eda_c_df['EDA_C'], 1.0, FS['chest'], 6)
    eda_data = nk.eda_phasic(nk.standardize(eda_df['EDA']), sampling_rate=FS['EDA'])
    eda_df['EDA_SCR'] = eda_data['EDA_Phasic']
    eda_df['EDA_SCL'] = eda_data['EDA_Tonic']
    eda_data_c = nk.eda_phasic(nk.standardize(eda_c_df['EDA_C']), sampling_rate=FS['chest'])
    eda_c_df['EDA_SCR_C'] = eda_data_c['EDA_Phasic']
    eda_c_df['EDA_SCL_C'] = eda_data_c['EDA_Tonic']

    for col in acc_df.columns:
        acc_df[col] = filter_signal_fir(acc_df[col].values)
    for col in acc_c_df.columns:
        acc_c_df[col] = filter_signal_fir(acc_c_df[col].values)

    n_700 = len(labels)
    df = pd.DataFrame(index=np.arange(n_700, dtype=np.int64))
    df['protocol_label'] = labels

    def add_signal(name, arr, fs, val_cols=None):
        arr = np.asarray(arr)
        idx = np.round(np.arange(len(arr), dtype=np.int64) * FS['label'] / fs).astype(np.int64)
        idx = np.clip(idx, 0, n_700 - 1)
        if arr.ndim == 1:
            df[name] = pd.Series(arr, index=idx).groupby(level=0).mean()
        else:
            for c_i, col_name in enumerate(val_cols):
                df[col_name] = pd.Series(arr[:, c_i], index=idx).groupby(level=0).mean()

    add_signal('EDA', eda_df['EDA'], FS['EDA'])
    add_signal('EDA_SCR', eda_df['EDA_SCR'], FS['EDA'])
    add_signal('EDA_SCL', eda_df['EDA_SCL'], FS['EDA'])
    add_signal('BVP', bvp_df['BVP'], FS['BVP'])
    add_signal('TEMP', temp_df['TEMP'], FS['TEMP'])
    add_signal('ACC', acc_df.values, FS['ACC'], ['ACC_x', 'ACC_y', 'ACC_z'])
    add_signal('ECG', ecg_df['ECG'], FS['chest'])
    add_signal('EDA_C', eda_c_df['EDA_C'], FS['chest'])
    add_signal('EDA_SCR_C', eda_c_df['EDA_SCR_C'], FS['chest'])
    add_signal('EDA_SCL_C', eda_c_df['EDA_SCL_C'], FS['chest'])
    add_signal('ACC_C', acc_c_df.values, FS['chest'], ['ACC_x_C', 'ACC_y_C', 'ACC_z_C'])
    add_signal('EMG_C', emg_c_df['EMG_C'], FS['chest'])
    add_signal('Resp_C', resp_c_df['Resp_C'], FS['chest'])
    add_signal('Temp_C', temp_c_df['Temp_C'], FS['chest'])

    df.reset_index(names='sample_index', inplace=True)
    return df

print('Signal preprocessing and STAI condition-label functions defined.')

Signal preprocessing and STAI condition-label functions defined.


## 3. Sliding Window Feature Extraction & Saving Pickles

In [4]:
def get_window_stats_ecg(data):
    if 'ECG' not in data or data['ECG'].dropna().empty:
        return {k: None for k in ['bpm', 'ibi', 'sdnn', 'sdsd', 'rmssd', 'pnn20', 'pnn50']}
    try:
        _, measures = hp.process(data['ECG'].dropna().reset_index(drop=True), 700)
        return {k: measures[k] for k in ['bpm', 'ibi', 'sdnn', 'sdsd', 'rmssd', 'pnn20', 'pnn50']}
    except Exception:
        return {k: None for k in ['bpm', 'ibi', 'sdnn', 'sdsd', 'rmssd', 'pnn20', 'pnn50']}


ACC_COLUMNS = ['ACC_x', 'ACC_y', 'ACC_z', 'ACC_x_C', 'ACC_y_C', 'ACC_z_C']


def _safe_acc_feature_values(series):
    series = pd.Series(series).dropna()
    if series.empty:
        return np.nan, np.nan, np.nan
    try:
        abs_integral = get_absolute_integral(series)
    except Exception:
        abs_integral = np.nan
    try:
        peak_freq = get_peak_freq(series)
    except Exception:
        peak_freq = np.nan
    return float(series.std()), abs_integral, peak_freq


def aggregate_acc_subwindows(window):
    """Aggregate ACC features from 5-second windows shifted by 0.25 seconds."""
    sub_len = int(round(FS['label'] * ACC_WINDOW_LENGTH_SECONDS))
    sub_step = int(round(FS['label'] * ACC_WINDOW_SHIFT_SECONDS))
    if sub_len <= 0 or sub_step <= 0:
        raise ValueError('ACC sub-window sample length and step must be positive.')

    subwindow_values = {f'{column}_{stat}': [] for column in ACC_COLUMNS for stat in ['std', 'abs_integral', 'peak_freq']}
    subwindow_count = 0
    for start in range(0, len(window) - sub_len + 1, sub_step):
        subwindow = window.iloc[start:start + sub_len]
        subwindow_count += 1
        for column in ACC_COLUMNS:
            std_value, integral_value, peak_value = _safe_acc_feature_values(subwindow[column])
            subwindow_values[f'{column}_std'].append(std_value)
            subwindow_values[f'{column}_abs_integral'].append(integral_value)
            subwindow_values[f'{column}_peak_freq'].append(peak_value)

    if subwindow_count == 0:
        raise ValueError('Main window is shorter than the configured ACC sub-window.')

    aggregated = {}
    for name, values in subwindow_values.items():
        numeric_values = np.asarray(values, dtype=float)
        aggregated[f'ACC_sub_{name}_mean'] = float(np.nanmean(numeric_values))
        aggregated[f'ACC_sub_{name}_std'] = float(np.nanstd(numeric_values))
    aggregated['ACC_subwindow_count'] = subwindow_count
    return aggregated


def extract_stai_features_subject(subject_id):
    df = compute_features_stai(subject_id)
    condition_labels = build_condition_labels(subject_id)
    samples = []
    window_len = int(round(FS['label'] * WINDOW_LENGTH_SECONDS))
    step = int(round(FS['label'] * WINDOW_SHIFT_SECONDS))
    feat_names = None
    candidate_windows = max(0, (len(df) - window_len) // step + 1)
    dropped_unlabeled_windows = 0
    dropped_ambiguous_windows = 0
    acc_subwindow_counts = []

    for start in range(0, len(df) - window_len + 1, step):
        end = start + window_len
        containing = [
            item for item in condition_labels
            if start >= item['start_seconds'] * FS['label']
            and end <= item['end_seconds'] * FS['label']
        ]
        if len(containing) != 1:
            if len(containing) > 1:
                dropped_ambiguous_windows += 1
            else:
                dropped_unlabeled_windows += 1
            continue
        condition = containing[0]
        window = df.iloc[start:end].drop(columns=['protocol_label', 'sample_index']).copy()

        net_acc = get_net_accel(window, 'wrist').rename('net_acc')
        window = pd.concat([net_acc, window], axis=1)
        net_acc_c = get_net_accel(window, 'chest').rename('net_acc_C')
        window = pd.concat([net_acc_c, window], axis=1)
        acc_features = aggregate_acc_subwindows(window)
        acc_subwindow_counts.append(acc_features['ACC_subwindow_count'])

        stats = get_window_stats(window, label=0)
        ecg_stats = get_window_stats_ecg(window)
        if 'label' in stats:
            del stats['label']
        x = pd.DataFrame(stats)
        if feat_names is None:
            feat_names = [f'{row}_{col}' for row in x.index for col in x.columns]
        wdf = pd.DataFrame(x.values.flatten()).T
        wdf.columns = feat_names

        wdf['binary_anxiety'] = condition['binary_anxiety']
        wdf['subject_id'] = subject_id
        wdf['condition'] = condition['condition']
        wdf['window_start_seconds'] = start / FS['label']
        wdf['window_end_seconds'] = end / FS['label']
        wdf['condition_stai_mean'] = condition['stai_mean']
        wdf['baseline_stai_mean'] = condition['baseline_stai_mean']
        for name, value in acc_features.items():
            wdf[name] = value

        wdf['BVP_peak_freq'] = get_peak_freq(window['BVP'].dropna())
        wdf['net_acc_abs_integral'] = get_absolute_integral(window['net_acc'].dropna())
        for col in ACC_COLUMNS:
            wdf[f'{col}_abs_integral'] = get_absolute_integral(window[col].dropna())
            series = window[col].dropna()
            wdf[f'{col}_peak_freq'] = None if series.empty else get_peak_freq(series)
        for key, value in ecg_stats.items():
            wdf[f'ECG_{key}'] = value
        wdf['EDA_slope'] = get_slope(window['EDA'].dropna())
        wdf['EDA_C_slope'] = get_slope(window['EDA_C'].dropna()) if window['EDA_C'].notna().any() else None
        wdf['EDA_drange'] = get_dynamic_range(window['EDA'].dropna())
        wdf['EDA_C_drange'] = get_dynamic_range(window['EDA_C'].dropna()) if window['EDA_C'].notna().any() else None
        wdf['EMG_drange'] = get_dynamic_range(window['EMG_C'].dropna())
        wdf['EMG_abs_integral'] = get_absolute_integral(window['EMG_C'].dropna())
        if window['Resp_C'].notna().any():
            resp_features = get_resp_features(window['Resp_C'].dropna())
            for r_name, r_val in zip([
                'Resp_C_rate', 'Resp_C_Inhal_mean', 'Resp_C_Inhal_std',
                'Resp_C_Exhal_mean', 'Resp_C_Exhal_std', 'Resp_C_I/E',
            ], resp_features):
                wdf[r_name] = r_val
        wdf['TEMP_drange'] = get_dynamic_range(window['TEMP'].dropna())
        wdf['TEMP_C_drange'] = get_dynamic_range(window['Temp_C'].dropna()) if window['Temp_C'].notna().any() else None
        wdf['TEMP_slope'] = get_slope(window['TEMP'].dropna())
        wdf['TEMP_C_slope'] = get_slope(window['Temp_C'].dropna()) if window['Temp_C'].notna().any() else None
        samples.append(wdf)

    result = pd.concat(samples, ignore_index=True) if samples else pd.DataFrame()
    result.attrs['candidate_windows'] = candidate_windows
    result.attrs['dropped_unlabeled_windows'] = dropped_unlabeled_windows
    result.attrs['dropped_ambiguous_windows'] = dropped_ambiguous_windows
    result.attrs['acc_subwindow_count'] = int(np.median(acc_subwindow_counts)) if acc_subwindow_counts else 0
    return result


# Process and save subjects to Data/Preprocessed n STAI-labeled/
summary = []
for subject_id in SUBJECTS:
    out_path = OUTPUT_DIR / f'S{subject_id}_stai_acc_subwindow.pkl'
    if out_path.exists() and SKIP_EXTRACTED_SUBJECTS:
        print(f'S{subject_id}: existing file skipped; ensure it was generated with this ACC sub-window version')
        continue
    if out_path.exists() and not OVERWRITE_EXISTING_OUTPUTS:
        raise FileExistsError(f'{out_path} exists and overwrite is disabled.')

    start_t = time.time()
    features_df = extract_stai_features_subject(subject_id)
    if features_df.empty:
        print(f'S{subject_id}: no fully-contained labeled windows; no file saved')
        continue
    if features_df.filter(regex=r'^ACC_sub_').isna().any().any():
        raise ValueError(f'S{subject_id}: ACC sub-window aggregation produced NaN values')
    features_df.to_pickle(out_path)
    summary.append({
        'subject': subject_id,
        'candidate_windows': features_df.attrs['candidate_windows'],
        'retained_windows': len(features_df),
        'dropped_unlabeled_windows': features_df.attrs['dropped_unlabeled_windows'],
        'dropped_ambiguous_windows': features_df.attrs['dropped_ambiguous_windows'],
        'acc_subwindows_per_main_window': features_df.attrs['acc_subwindow_count'],
        'n_features': len(features_df.columns),
        'no_anxiety_windows': int((features_df['binary_anxiety'] == 0).sum()),
        'anxiety_windows': int((features_df['binary_anxiety'] == 1).sum()),
        'saved_to': out_path.name,
        'seconds': round(time.time() - start_t, 2),
    })
    print(summary[-1])
    del features_df
    gc.collect()

pd.DataFrame(summary)

{'subject': 2, 'candidate_windows': 502, 'retained_windows': 131, 'dropped_unlabeled_windows': 371, 'dropped_ambiguous_windows': 0, 'acc_subwindows_per_main_window': 221, 'n_features': 161, 'no_anxiety_windows': 131, 'anxiety_windows': 0, 'saved_to': 'S2_stai_acc_subwindow.pkl', 'seconds': 215.85}
{'subject': 3, 'candidate_windows': 537, 'retained_windows': 133, 'dropped_unlabeled_windows': 404, 'dropped_ambiguous_windows': 0, 'acc_subwindows_per_main_window': 221, 'n_features': 161, 'no_anxiety_windows': 55, 'anxiety_windows': 78, 'saved_to': 'S3_stai_acc_subwindow.pkl', 'seconds': 369.98}
{'subject': 4, 'candidate_windows': 531, 'retained_windows': 137, 'dropped_unlabeled_windows': 394, 'dropped_ambiguous_windows': 0, 'acc_subwindows_per_main_window': 221, 'n_features': 161, 'no_anxiety_windows': 59, 'anxiety_windows': 78, 'saved_to': 'S4_stai_acc_subwindow.pkl', 'seconds': 207.92}
{'subject': 5, 'candidate_windows': 517, 'retained_windows': 138, 'dropped_unlabeled_windows': 379, 'dr

,subject,candidate_windows,retained_windows,dropped_unlabeled_windows,dropped_ambiguous_windows,acc_subwindows_per_main_window,n_features,no_anxiety_windows,anxiety_windows,saved_to,seconds
0,2,502,131,371,0,221,161,131,0,S2_stai_acc_subwindow.pkl,215.85
1,3,537,133,404,0,221,161,55,78,S3_stai_acc_subwindow.pkl,369.98
2,4,531,137,394,0,221,161,59,78,S4_stai_acc_subwindow.pkl,207.92
3,5,517,138,379,0,221,161,138,0,S5_stai_acc_subwindow.pkl,201.49
4,6,585,137,448,0,221,161,135,2,S6_stai_acc_subwindow.pkl,220.50
5,7,432,139,293,0,221,161,88,51,S7_stai_acc_subwindow.pkl,287.30
6,8,451,133,318,0,221,161,133,0,S8_stai_acc_subwindow.pkl,167.16
7,9,431,134,297,0,221,161,134,0,S9_stai_acc_subwindow.pkl,196.36
8,10,454,144,310,0,221,161,144,0,S10_stai_acc_subwindow.pkl,245.15
9,11,432,144,288,0,221,161,90,54,S11_stai_acc_subwindow.pkl,239.10


## 4. Preview a Regenerated STAI-Labeled Subject Dataset

Select a subject pickle, inspect its binary label distribution and STAI provenance, and preview rows.

In [5]:
PREVIEW_SUBJECT = 2
preview_path = OUTPUT_DIR / f'S{PREVIEW_SUBJECT}_stai.pkl'
if not preview_path.exists():
    raise FileNotFoundError(f'Dataset not found: {preview_path}. Run Section 3 first.')

preview_df = pd.read_pickle(preview_path)
if 'condition_stai_mean' not in preview_df.columns:
    raise ValueError('This pickle lacks STAI provenance and appears to be an old protocol-labeled output. Regenerate it with Section 3.')
print(f'File: {preview_path.name}; shape: {preview_df.shape}')
print('Binary labels (0=No Anxiety, 1=Anxiety):')
display(preview_df['binary_anxiety'].value_counts(dropna=False).sort_index().rename_axis('binary_anxiety').to_frame('windows'))
print('Per-condition STAI means and assigned labels:')
display(preview_df[['condition', 'baseline_stai_mean', 'condition_stai_mean', 'binary_anxiety']].drop_duplicates().sort_values('condition'))
display(preview_df.head())

File: S2_stai.pkl; shape: (131, 188)
Binary labels (0=No Anxiety, 1=Anxiety):


,windows
binary_anxiety,
0,103
1,28


Per-condition STAI means and assigned labels:


,condition,baseline_stai_mean,condition_stai_mean,binary_anxiety
77,Fun,3.0,3.2,1
48,Medi 1,3.0,1.2,0
103,Medi 2,3.0,1.8,0
0,TSST,3.0,1.6,0
46,sRead,3.0,3.2,1


,net_acc_C_mean,net_acc_C_std,net_acc_C_min,net_acc_C_max,net_acc_mean,net_acc_std,net_acc_min,net_acc_max,EDA_mean,EDA_std,...,Resp_C_rate,Resp_C_Inhal_mean,Resp_C_Inhal_std,Resp_C_Exhal_mean,Resp_C_Exhal_std,Resp_C_I/E,TEMP_drange,TEMP_C_drange,TEMP_slope,TEMP_C_slope
0,0.930242,0.008693,0.859829,0.982168,62.485828,0.557542,60.431757,63.233511,0.594088,0.019942,...,0.191606,2.636429,0.982913,2.999429,1.219457,0.878977,1.003553,1.011093,-0.000429,1.618075e-06
1,0.930223,0.010815,0.859829,1.001470,62.460816,0.543056,60.431757,63.262222,0.592131,0.027559,...,0.204072,2.589481,1.170803,2.867571,1.281923,0.993324,1.004148,1.011566,-0.000525,3.233636e-06
2,0.929467,0.011545,0.859829,1.001470,62.428783,0.533283,60.431757,63.262222,0.605914,0.044586,...,0.191891,2.828571,1.212032,2.770714,1.192616,1.020882,1.004148,1.010899,-0.000507,3.621997e-06
3,0.928671,0.012481,0.859829,1.001470,62.379929,0.595385,60.411404,63.262222,0.620791,0.049391,...,0.193599,2.824286,1.261812,2.836714,1.217616,0.995619,1.005340,1.011751,-0.000459,2.790777e-06
4,0.928120,0.012512,0.848716,1.001470,62.495687,0.533677,60.411404,63.262222,0.642464,0.054240,...,0.196125,3.035195,1.228791,2.388857,0.941385,1.397620,1.005349,1.009990,-0.000562,5.198775e-07


## 5. Compare Against the Forked Subject Features

This is a structural comparison only: the forked CSV has WESAD protocol targets, whereas this dataset has STAI-baseline-relative targets. They are different targets and feature-extraction paths, so their target counts or row-wise values are not expected to match.

In [6]:
COMPARISON_SUBJECT = PREVIEW_SUBJECT
stai_path = OUTPUT_DIR / f'S{COMPARISON_SUBJECT}_stai.pkl'
forked_path = REPO_ROOT / '_forked' / 'data' / 'subject_feats' / f'S{COMPARISON_SUBJECT}_feats.csv'
if not stai_path.exists() or not forked_path.exists():
    raise FileNotFoundError(f'Missing comparison input: {stai_path} or {forked_path}')

stai_compare_df = pd.read_pickle(stai_path)
forked_compare_df = pd.read_csv(forked_path, index_col=0)
if 'condition_stai_mean' not in stai_compare_df.columns:
    raise ValueError('STAI comparison pickle is stale; regenerate it with Section 3.')

stai_features = set(stai_compare_df.columns) - {
    'binary_anxiety', 'subject_id', 'condition', 'window_start_seconds',
    'window_end_seconds', 'condition_stai_mean', 'baseline_stai_mean',
}
forked_targets = [c for c in forked_compare_df.columns if str(c) in {'0', '1', '2'}]
forked_features = set(forked_compare_df.columns) - set(forked_targets)
common_features = sorted(stai_features & forked_features)
print(f'Subject S{COMPARISON_SUBJECT}: STAI={stai_compare_df.shape}, forked={forked_compare_df.shape}')
print(f'Common feature names: {len(common_features)}')
print(f'STAI label counts:\n{stai_compare_df.binary_anxiety.value_counts().sort_index()}')
print('Shared names do not imply identical numerical features or equivalent labels.')
if common_features:
    display(pd.DataFrame({
        'stai_mean': stai_compare_df[common_features].apply(pd.to_numeric, errors='coerce').mean(),
        'forked_mean': forked_compare_df[common_features].apply(pd.to_numeric, errors='coerce').mean(),
    }))

Subject S2: STAI=(131, 188), forked=(37, 121)
Common feature names: 117
STAI label counts:
0    103
1     28
Name: binary_anxiety, dtype: int64
Shared names do not imply identical numerical features or equivalent labels.


,stai_mean,forked_mean
ACC_x_C_abs_integral,32148.070613,20.006679
ACC_x_C_max,0.882477,0.000566
ACC_x_C_mean,0.765430,0.000517
ACC_x_C_min,0.660699,0.000478
ACC_x_C_peak_freq,0.003718,0.003365
...,...,...
net_acc_abs_integral,120693.299916,48.921033
net_acc_max,64.204907,0.048108
net_acc_mean,62.861094,0.027664
net_acc_min,58.234764,0.010544


## 6. Binary Classification Model Training & Evaluation

Train and evaluate models from the generated `S*_stai.pkl` feature datasets in `Data/Preprocessed n STAI-labeled/`.

In [7]:
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GroupShuffleSplit
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# Training specifications — edit these values before running this cell.
TEST_SIZE = 0.2
RANDOM_STATE = 42
RF_N_ESTIMATORS = 100
SVM_KERNEL = 'rbf'
XGB_N_ESTIMATORS = 100  # boosting rounds, not epochs
KNN_N_NEIGHBORS = 5

pkl_files = sorted(OUTPUT_DIR.glob('S*_stai.pkl'))
if not pkl_files:
    raise FileNotFoundError(f'No S*_stai.pkl datasets found in {OUTPUT_DIR}. Run Section 3 first.')

all_dfs = []
for path in pkl_files:
    subject_df = pd.read_pickle(path)
    if 'binary_anxiety' not in subject_df.columns or 'subject_id' not in subject_df.columns:
        raise ValueError(f'{path.name} is missing binary_anxiety or subject_id; regenerate it with the current STAI pipeline.')
    all_dfs.append(subject_df)

combined_df = pd.concat(all_dfs, ignore_index=True)
combined_df = combined_df[combined_df['binary_anxiety'].isin([0, 1])].copy()

# Do not feed labels or questionnaire/provenance metadata into the predictors.
metadata_columns = {
    'binary_anxiety', 'subject_id', 'condition', 'window_start_seconds',
    'window_end_seconds', 'condition_stai_mean', 'baseline_stai_mean',
}
feature_columns = [column for column in combined_df.columns if column not in metadata_columns]
X = combined_df[feature_columns].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)
y = combined_df['binary_anxiety'].astype(int)
groups = combined_df['subject_id'].astype(int)

class_counts = y.value_counts().sort_index()
print(f'Loaded {len(pkl_files)} STAI-labeled subjects; combined shape={combined_df.shape}')
print('Class distribution (0=No Anxiety, 1=Anxiety):')
display(class_counts.rename_axis('binary_anxiety').to_frame('windows'))
if y.nunique() != 2:
    raise ValueError(f'Both binary classes are required; got {class_counts.to_dict()}')
if groups.nunique() < 2:
    raise ValueError('At least two subjects are required for subject-grouped evaluation.')

splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_indices, test_indices = next(splitter.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_indices], X.iloc[test_indices]
y_train, y_test = y.iloc[train_indices], y.iloc[test_indices]
train_subjects = set(groups.iloc[train_indices])
test_subjects = set(groups.iloc[test_indices])
print(f'Train: {X_train.shape}; test: {X_test.shape}')
print(f'Train subjects: {sorted(train_subjects)}; test subjects: {sorted(test_subjects)}')
if train_subjects & test_subjects:
    raise RuntimeError('Subject leakage detected across train/test split.')
if y_train.nunique() != 2 or y_test.nunique() != 2:
    raise ValueError('Grouped split must contain both classes in train and test; adjust TEST_SIZE or the subject grouping.')
if KNN_N_NEIGHBORS > len(X_train):
    raise ValueError('KNN_N_NEIGHBORS exceeds number of training windows.')

models = {
    'Random Forest': RandomForestClassifier(n_estimators=RF_N_ESTIMATORS, random_state=RANDOM_STATE),
    'SVM': SVC(kernel=SVM_KERNEL, random_state=RANDOM_STATE),
    'XGBoost': XGBClassifier(n_estimators=XGB_N_ESTIMATORS, eval_metric='logloss', random_state=RANDOM_STATE),
    'KNN': KNeighborsClassifier(n_neighbors=KNN_N_NEIGHBORS),
}

results = {}
for name, model in models.items():
    print(f'\n--- Training {name} ---')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results[name] = {
        'model': model,
        'accuracy': accuracy_score(y_test, y_pred),
        'confusion_matrix': confusion_matrix(y_test, y_pred, labels=[0, 1]),
        'classification_report': classification_report(
            y_test, y_pred, labels=[0, 1], target_names=['No Anxiety', 'Anxiety'], zero_division=0
        ),
    }
    print(f"{name} Accuracy: {results[name]['accuracy']:.4f}")
    print('Confusion Matrix (rows=true, columns=predicted; [0, 1]):\n', results[name]['confusion_matrix'])
    print('Classification Report:\n', results[name]['classification_report'])
    model_filename = MODELS_DIR / f'stai_baseline_relative_{name.lower().replace(" ", "_")}_model.joblib'
    joblib.dump(model, model_filename)
    print(f'Saved model to {model_filename}')

print('\nSTAI-baseline-relative models trained and evaluated.')

Loaded 15 STAI-labeled subjects; combined shape=(2078, 188)
Class distribution (0=No Anxiety, 1=Anxiety):


,windows
binary_anxiety,
0,1708
1,370


Train: (1660, 181); test: (418, 181)
Train subjects: [3, 4, 5, 6, 7, 8, 9, 10, 13, 15, 16, 17]; test subjects: [2, 11, 14]

--- Training Random Forest ---
Random Forest Accuracy: 0.6627
Confusion Matrix (rows=true, columns=predicted; [0, 1]):
 [[277   0]
 [141   0]]
Classification Report:
               precision    recall  f1-score   support

  No Anxiety       0.66      1.00      0.80       277
     Anxiety       0.00      0.00      0.00       141

    accuracy                           0.66       418
   macro avg       0.33      0.50      0.40       418
weighted avg       0.44      0.66      0.53       418

Saved model to e:\Rizqy\Kuliah\Skripsi\repo-skripsi\Models\stai_baseline_relative_random_forest_model.joblib

--- Training SVM ---
SVM Accuracy: 0.6029
Confusion Matrix (rows=true, columns=predicted; [0, 1]):
 [[228  49]
 [117  24]]
Classification Report:
               precision    recall  f1-score   support

  No Anxiety       0.66      0.82      0.73       277
     Anxiety    